# Data Science Mini Project: Breast Cancer Diagnosis Analysis
**Module 5 (Day 17–20)**

**Dataset:** Breast Cancer Wisconsin (Diagnostic), a real-world public dataset from the UCI Machine Learning Repository (also on Kaggle). It has 569 patients and 30 measurements of cell nuclei from biopsy images, labelled **malignant** or **benign**.

**Goal:** find which measurements best separate malignant from benign tumours, visualize the patterns, and check how well a simple model can predict the diagnosis.

**Workflow:** Load → Clean → Explore → Visualize → Insights → Model → Conclusion

## 1. Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style='whitegrid')

data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
# In the original data 0 = malignant, 1 = benign. Make it readable.
df['diagnosis'] = df['target'].map({0: 'Malignant', 1: 'Benign'})
df = df.drop(columns='target')
print('Shape:', df.shape)
df.head()

## 2. Clean & check quality

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
print(df['diagnosis'].value_counts())
print((df['diagnosis'].value_counts(normalize=True) * 100).round(1).astype(str) + '%')
df.describe().T.head(10).round(2)

## 3. Explore & visualize

In [ ]:
# Class balance
plt.figure(figsize=(5, 4))
ax = sns.countplot(data=df, x='diagnosis', hue='diagnosis', palette={'Malignant': '#e45756', 'Benign': '#4c78a8'}, legend=False)
for c in ax.containers: ax.bar_label(c)
plt.title('Diagnosis Count'); plt.xlabel(''); plt.tight_layout(); plt.show()

In [ ]:
# Distribution of key features by diagnosis
feats = ['mean radius', 'mean area', 'mean concavity', 'mean texture']
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, f in zip(axes.ravel(), feats):
    sns.histplot(data=df, x=f, hue='diagnosis', kde=True, ax=ax, palette={'Malignant': '#e45756', 'Benign': '#4c78a8'})
    ax.set_title(f'{f.title()} by Diagnosis')
plt.tight_layout(); plt.show()

In [ ]:
# Box plots: spread of selected features
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, f in zip(axes, ['mean radius', 'mean concave points', 'worst perimeter']):
    sns.boxplot(data=df, x='diagnosis', y=f, hue='diagnosis', ax=ax,
                palette={'Malignant': '#e45756', 'Benign': '#4c78a8'}, legend=False)
    ax.set_title(f.title()); ax.set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
# Scatter: two strongest features together
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x='mean radius', y='mean concave points', hue='diagnosis',
                palette={'Malignant': '#e45756', 'Benign': '#4c78a8'}, alpha=0.75)
plt.title('Radius vs Concave Points'); plt.tight_layout(); plt.show()

## 4. Correlation & key drivers

In [ ]:
num = df.drop(columns='diagnosis')
is_malignant = (df['diagnosis'] == 'Malignant').astype(int)
corr_target = num.corrwith(is_malignant).sort_values(key=abs, ascending=False)
top = corr_target.head(10)
print('Top 10 features related to malignancy:\n', top.round(2))

plt.figure(figsize=(7, 4.5))
top.sort_values().plot(kind='barh', color=['#e45756' if v > 0 else '#4c78a8' for v in top.sort_values()])
plt.title('Correlation with Malignant Diagnosis'); plt.xlabel('Correlation'); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(num[list(top.index[:8])].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Between Top Features'); plt.tight_layout(); plt.show()

In [ ]:
# Group comparison
summary = df.groupby('diagnosis')[['mean radius', 'mean area', 'mean concavity', 'mean concave points']].mean().round(3)
summary.loc['Ratio (Malignant / Benign)'] = (summary.loc['Malignant'] / summary.loc['Benign']).round(2)
summary

## 5. Predictive model
A quick check of whether these measurements are enough to predict the diagnosis.

In [ ]:
X = num
y = is_malignant
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)

logreg = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)

p_lr = logreg.predict(scaler.transform(X_test))
p_rf = rf.predict(X_test)
print('Logistic Regression accuracy:', round(accuracy_score(y_test, p_lr), 3))
print('Random Forest accuracy      :', round(accuracy_score(y_test, p_rf), 3))
print('\nRandom Forest report:\n', classification_report(y_test, p_rf, target_names=['Benign', 'Malignant']))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(confusion_matrix(y_test, p_rf), annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['Benign', 'Malignant'], yticklabels=['Benign', 'Malignant'])
ax[0].set_title('Confusion Matrix (Random Forest)'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual')
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(8)
imp.plot(kind='barh', ax=ax[1], color='teal'); ax[1].set_title('Top Feature Importances')
plt.tight_layout(); plt.show()

In [ ]:
# Data-driven insights summary (values update automatically)
mal = df[df['diagnosis'] == 'Malignant']; ben = df[df['diagnosis'] == 'Benign']
print(f"1. {len(mal)/len(df):.0%} of cases are malignant ({len(mal)} of {len(df)}).")
print(f"2. Malignant tumours have a {mal['mean radius'].mean()/ben['mean radius'].mean():.2f}x larger mean radius and "
      f"{mal['mean area'].mean()/ben['mean area'].mean():.2f}x larger mean area than benign ones.")
print(f"3. Strongest single indicator: '{top.index[0]}' (correlation {top.iloc[0]:.2f}).")
print(f"4. Best model accuracy on unseen patients: {max(accuracy_score(y_test, p_lr), accuracy_score(y_test, p_rf)):.1%}.")

## 6. Key Insights
- **Size matters:** malignant tumours have a clearly larger radius, perimeter and area than benign ones.
- **Shape matters:** concavity and concave points are among the strongest indicators of malignancy; irregular, indented nuclei are more likely malignant.
- **Many features move together:** radius, perimeter and area are highly correlated, so they carry overlapping information.
- **Prediction works well:** simple models reach roughly 95–97% accuracy on unseen data, showing these measurements separate the two groups clearly.
- **Limits:** this is a small dataset and accuracy alone is not enough; in real screening, missing a malignant case (false negative) is the costliest error, so recall for the malignant class matters most. This is a learning project, not medical advice.

## 7. Conclusion
I cleaned and explored a real-world medical dataset, visualized how diagnosis relates to cell measurements, found the strongest indicators, and built models that predict diagnosis with high accuracy. Data analysis, visualization and machine learning combined turn raw numbers into useful insights.